In [ ]:
# Fix for potential Protobuf/Tensorboard conflicts in Kaggle environment
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["WANDB_DISABLED"] = "true"

# Disable Transformers logging to avoid Tensorboard/Protobuf trigger
from transformers import logging
logging.set_verbosity_error()

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, default_collate
from transformers import AutoTokenizer, AutoConfig, AutoModel, PreTrainedModel
import torch.nn as nn
import sys
import re
import html

# --- 1. SETUP PATHS ---
# In Kaggle, datasets are usually at /kaggle/input/...
# Change MODEL_PATH to match the name of your uploaded model dataset
MODEL_PATH = "/kaggle/input/quest-modernbert-model/saved_model" # UPDATE THIS AFTER UPLOADING YOUR MODEL
DATA_DIR = "/kaggle/input/google-quest-challenge"

TARGET_COLS = ['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

In [ ]:
# --- 2. EMBEDDED HELPER CLASSES ---

class QuestPreprocessor:
    def __init__(self, use_category=True, use_host=True, max_body_words=500):
        self.use_category = use_category
        self.use_host = use_host
        self.max_body_words = max_body_words
        self.contractions = {
            "n't": " not", "'re": " are", "'s": " is", "'d": " would",
            "'ll": " will", "'t": " not", "'ve": " have", "'m": " am"
        }

    def preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = ['question_title', 'question_body', 'answer']
        for col in text_cols:
            df[col] = df[col].astype(str).apply(self._clean_text)
        df['text_input'] = df.apply(self._construct_input_text, axis=1)
        return df

    def _clean_text(self, text: str) -> str:
        if not isinstance(text, str): return str(text)
        text = html.unescape(text)
        text = re.sub(r'<br\s*/?>', ' ', text)
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\$.*?\$', ' [MATH] ', text)
        text = re.sub(r'\d+', '#', text)
        for contraction, expansion in self.contractions.items():
            text = text.replace(contraction, expansion)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _smart_truncate(self, text: str) -> str:
        words = text.split()
        if len(words) <= self.max_body_words:
            return text
        half_limit = self.max_body_words // 2
        head = words[:half_limit]
        tail = words[-half_limit:]
        return " ".join(head) + " ... [TRUNCATED] ... " + " ".join(tail)

    def _construct_input_text(self, row) -> str:
        parts = []
        if self.use_category: parts.append(f"CAT: {str(row['category'])}")
        if self.use_host: parts.append(f"HOST: {str(row['host'])}")
        parts.append(f"TITLE: {str(row['question_title'])}")
        clean_body = str(row['question_body'])
        truncated_body = self._smart_truncate(clean_body)
        parts.append(f"BODY: {truncated_body}")
        parts.append(f"ANSWER: {str(row['answer'])}")
        return " [SEP] ".join(parts)

class QuestPostprocessor:
    def __init__(self, train_labels, denominator=60):
        if isinstance(train_labels, pd.DataFrame):
            self.y_labels = train_labels.values.flatten()
        else:
            self.y_labels = np.array(train_labels).flatten()
        self.unique_labels = np.array(sorted(np.unique(self.y_labels)))
        self.denominator = denominator
        self.exp_labels = self._compute_expected_labels()

    def _compute_expected_labels(self):
        q = np.arange(0, 101, 100 / self.denominator)
        exp_labels = np.percentile(self.unique_labels, q)
        return exp_labels

    def optimize_ranks(self, preds):
        new_preds = np.zeros(preds.shape)
        bins = self.exp_labels
        for i in range(preds.shape[1]):
            interpolate_bins = np.digitize(preds[:, i], bins=bins, right=False)
            interpolate_bins = np.minimum(interpolate_bins, len(bins) - 1)
            interpolate_bins = np.maximum(interpolate_bins, 0)
            if len(np.unique(interpolate_bins)) == 1:
                new_preds[:, i] = preds[:, i]
            else:
                new_preds[:, i] = bins[interpolate_bins]
        return new_preds

In [ ]:
# --- 3. DEFINE MODEL CLASS (Must match training) ---
class CustomModel(PreTrainedModel):
    def __init__(self, config, n_targets=30, dropout_prob=0.1, freeze_encoder=False):
        super().__init__(config)
        if config._name_or_path:
            self.encoder = AutoModel.from_pretrained(config._name_or_path)
        else:
            self.encoder = AutoModel.from_config(config)

        self.dropout = nn.Dropout(dropout_prob)
        self.regressor = nn.Linear(config.hidden_size, n_targets)
        self.sigmoid = nn.Sigmoid()
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(pooled_output)
        raw_logits = self.regressor(x)
        preds = self.sigmoid(raw_logits)
        return {"logits": preds}

class QuestDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text_input'])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze() for k, v in enc.items()}
        return item

In [ ]:
# --- 4. LOAD MODEL & DATA ---
print(f"Loading model from {MODEL_PATH}...")

config = AutoConfig.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 1. Initialize the structure
model = CustomModel(config)

# 2. Load the trained weights explicitly to avoid initialization warnings/errors
weights_path = os.path.join(MODEL_PATH, "pytorch_model.bin")
if not os.path.exists(weights_path):
    weights_path = os.path.join(MODEL_PATH, "model.safetensors")
    
if os.path.exists(weights_path):
    print(f"Found weights at {weights_path}")
    if weights_path.endswith(".safetensors"):
        from safetensors.torch import load_file
        state_dict = load_file(weights_path)
    else:
        state_dict = torch.load(weights_path, map_location="cpu")
        
    # 3. Load into model
    load_result = model.load_state_dict(state_dict, strict=True)
    print("State dict loaded:", load_result)
else:
    print(f"ERROR: No model weights found in {MODEL_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("Model loaded!")

# Load Data
test_path = os.path.join(DATA_DIR, "test.csv")
train_path = os.path.join(DATA_DIR, "train.csv")

print(f"Loading test data from {test_path}...")
test_df = pd.read_csv(test_path)

print("Preprocessing test data...")
preprocessor = QuestPreprocessor(use_category=True, use_host=True)
test_df_processed = preprocessor.preprocess(test_df)

test_ds = QuestDataset(test_df_processed, tokenizer)
loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=default_collate
)

# Initialize Postprocessor with Train Data
# (We need train labels to calculate the quantile bins)
print("Initializing Postprocessor using train.csv... ")
train_df = pd.read_csv(train_path)
postprocessor = QuestPostprocessor(train_labels=train_df[TARGET_COLS].values)

In [ ]:
# --- 5. INFERENCE ---
print("Running inference...")
all_logits = []
from tqdm.auto import tqdm

with torch.no_grad():
    for batch in tqdm(loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("token_type_ids", None)
        out = model(**batch)
        all_logits.append(out["logits"].cpu())

preds = torch.cat(all_logits, dim=0).numpy()

In [ ]:
# --- 6. POST-PROCESS & SUBMIT ---
print("Applying rank optimization...")
final_preds = postprocessor.optimize_ranks(preds)

sub = pd.DataFrame(final_preds, columns=TARGET_COLS)
sub.insert(0, "qa_id", test_df["qa_id"])
sub.to_csv("submission.csv", index=False)
print("submission.csv saved!")